I had to make some manual edits to missing entries from the Python-processed dataset; saved this as a tab delimited file, this notebook converts it back to geojson

In [3]:
import json
import os
from pathlib import Path
import pandas as pd

In [5]:
data = pd.read_csv("manual_updates_manifests.txt", sep="\t")

In [6]:
data

,name,related_artefacts,geometry_type,coordinates,manifest
0,Grand Causeway,https://digital.library.leeds.ac.uk/15933/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
1,Grand Causeway,https://digital.library.leeds.ac.uk/15941/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
2,Grand Causeway,https://digital.library.leeds.ac.uk/15944/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
3,Grand Causeway,https://digital.library.leeds.ac.uk/15930/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
4,Grand Causeway,https://digital.library.leeds.ac.uk/15925/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
5,Grand Causeway,https://digital.library.leeds.ac.uk/15940/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
6,Grand Causeway,https://digital.library.leeds.ac.uk/15931/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
7,Grand Causeway,https://digital.library.leeds.ac.uk/15929/,Polygon,"[[[-6.512022, 55.241873], [-6.512156, 55.24181...",https://iiif.library.leeds.ac.uk/presentation/...
8,Portnaboe,https://digital.library.leeds.ac.uk/15962/,Polygon,"[[[-6.521061, 55.236523], [-6.520437, 55.23601...",https://iiif.library.leeds.ac.uk/presentation/...
9,Amphitheatre,https://digital.library.leeds.ac.uk/15936/,Polygon,"[[[-6.504895, 55.244388], [-6.503879, 55.24387...",https://iiif.library.leeds.ac.uk/presentation/...


In [7]:
# Convert to GeoJSON
features = []
for idx, row in data.iterrows():
    feature = {
        "type": "Feature",
        "geometry": {
            "type": row['geometry_type'],
            "coordinates": json.loads(row['coordinates'])
        },
        "properties": {
            "name": row['name'],
            "related_artefacts": row['related_artefacts'],
            "manifest": row['manifest']
        }
    }
    features.append(feature)

geojson_data = {
    "type": "FeatureCollection",
    "features": features
}

# Save to file
output_file = "manual_updates_manifests.geojson"
with open(output_file, 'w') as f:
    json.dump(geojson_data, f, indent=2)


In [8]:
# Consolidate: group by name and aggregate related_artefacts and manifest into lists
data_consolidated = data.groupby('name', as_index=False).agg({
    'geometry_type': 'first',  # All rows with same name should have same geometry
    'coordinates': 'first',    # All rows with same name should have same coordinates
    'related_artefacts': list,  # Collect all URLs into a list
    'manifest': list            # Collect all manifest URLs into a list
})

print(f"Original: {len(data)} rows")
print(f"Consolidated: {len(data_consolidated)} rows")
print("\nFirst few rows:")
data_consolidated.head()

Original: 38 rows
Consolidated: 15 rows

First few rows:


,name,geometry_type,coordinates,related_artefacts,manifest
0,Amphitheatre,Polygon,"[[[-6.504895, 55.244388], [-6.503879, 55.24387...","[https://digital.library.leeds.ac.uk/15936/, h...",[https://iiif.library.leeds.ac.uk/presentation...
1,Chimney Stacks,Polygon,"[[[-6.504498, 55.246767], [-6.504426, 55.24656...","[https://digital.library.leeds.ac.uk/15956/, h...",[https://iiif.library.leeds.ac.uk/presentation...
2,Giant's Gate,Point,"[-6.51081, 55.240003]",[https://digital.library.leeds.ac.uk/15935/],[https://iiif.library.leeds.ac.uk/presentation...
3,Giant's Loom,Point,"[-6.511229, 55.240851]","[https://digital.library.leeds.ac.uk/15932/, h...",[https://iiif.library.leeds.ac.uk/presentation...
4,Giant's Well,Point,"[-6.51258, 55.240108]",[https://digital.library.leeds.ac.uk/15947/],[https://iiif.library.leeds.ac.uk/presentation...


In [9]:
# Save consolidated data to GeoJSON
features_consolidated = []
for idx, row in data_consolidated.iterrows():
    feature = {
        "type": "Feature",
        "geometry": {
            "type": row['geometry_type'],
            "coordinates": json.loads(row['coordinates'])
        },
        "properties": {
            "name": row['name'],
            "related_artefacts": row['related_artefacts'],  # Now a list
            "manifest": row['manifest']  # Now a list
        }
    }
    features_consolidated.append(feature)

geojson_consolidated = {
    "type": "FeatureCollection",
    "features": features_consolidated
}

# Save to file
output_file_consolidated = "manual_updates_manifests_consolidated.geojson"
with open(output_file_consolidated, 'w') as f:
    json.dump(geojson_consolidated, f, indent=2)
